# Fit Conjoint Model From `laptops.csv`

This notebook reads a `laptops.csv` file, rebuilds the conjoint buckets from raw values, fits the overall model and company-specific models, and exports plots plus CSV outputs for betas, part-worths, zero-centered part-worths, and attribute importance.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
CSV_CANDIDATES = [
    Path('/content/flipkart_laptops.csv'),
    Path('flipkart_laptops.csv')
]
CSV_PATH = next((p for p in CSV_CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('Upload laptops.csv to /content or place it next to this notebook.')

OUTPUT_DIR = Path('/content/conjoint_fit_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_BRAND_COUNT = 10
MAJOR_BRAND_COUNT = 50
ALLOWED_COMPANIES = ['ASUS', 'Lenovo', 'HP', 'DELL', 'Acer']
COMPANY_NAME_MAP = {
    'asus': 'ASUS',
    'lenovo': 'Lenovo',
    'hp': 'HP',
    'hewlett packard': 'HP',
    'dell': 'DELL',
    'acer': 'Acer',
}

CORE_ATTRIBUTES_OVERALL = [
    'Company_Bucket_10',
    'Price_Bucket',
    'Processor_Tier',
    'RAM_Bucket',
    'Storage_Bucket',
    'Warranty_Bucket',
]

CORE_ATTRIBUTES_WITHIN_BRAND = [
    'Price_Bucket',
    'Processor_Tier',
    'RAM_Bucket',
    'Storage_Bucket',
    'Warranty_Bucket',
]

TARGET_COL = 'Star_Rating_num'

print('Using CSV:', CSV_PATH)
print('Outputs will be written to:', OUTPUT_DIR)

In [ ]:
def norm(x):
    if pd.isna(x):
        return ''
    return str(x).strip()


def is_missing(x):
    return norm(x) in {'', 'N/A', 'NA', 'None', 'null', 'NULL', '-'}


def parse_float(x):
    x = norm(x)
    if is_missing(x):
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan


def parse_int(x):
    x = norm(x)
    if is_missing(x):
        return np.nan
    try:
        return int(float(x))
    except Exception:
        return np.nan


def parse_ram_gb(x):
    x = norm(x).lower()
    match = re.search(r'(\d+)\s*gb', x)
    return int(match.group(1)) if match else np.nan


def parse_storage_gb(x):
    x = norm(x)
    if is_missing(x):
        return np.nan
    try:
        value = float(x)
    except Exception:
        return np.nan
    return int(value * 1024) if value <= 10 else int(value)


def bucket_price(x):
    value = parse_int(x)
    if pd.isna(value):
        return 'Missing'
    if value < 30000:
        return 'Budget'
    if value < 60000:
        return 'Mid-range'
    return 'Premium'


def bucket_processor_family(x):
    s = norm(x).lower()
    if is_missing(s):
        return 'Missing'
    if 'celeron' in s:
        return 'Intel Celeron'
    if 'pentium' in s:
        return 'Intel Pentium'
    if 'core ultra 9' in s:
        return 'Intel Core Ultra 9'
    if 'core ultra 7' in s:
        return 'Intel Core Ultra 7'
    if 'core ultra 5' in s:
        return 'Intel Core Ultra 5'
    if 'core 9' in s and 'ultra' not in s:
        return 'Intel Core 9'
    if 'core 7' in s and 'ultra' not in s:
        return 'Intel Core 7'
    if 'core 5' in s and 'ultra' not in s:
        return 'Intel Core 5'
    if 'core 3' in s and 'ultra' not in s:
        return 'Intel Core 3'
    if 'core i9' in s:
        return 'Intel Core i9'
    if 'core i7' in s:
        return 'Intel Core i7'
    if 'core i5' in s:
        return 'Intel Core i5'
    if 'core i3' in s:
        return 'Intel Core i3'
    if 'ryzen ai 9' in s:
        return 'AMD Ryzen AI 9'
    if 'ryzen ai 7' in s:
        return 'AMD Ryzen AI 7'
    if 'ryzen 9' in s:
        return 'AMD Ryzen 9'
    if 'ryzen 7' in s:
        return 'AMD Ryzen 7'
    if 'ryzen 5' in s:
        return 'AMD Ryzen 5'
    if 'ryzen 3' in s:
        return 'AMD Ryzen 3'
    if 'athlon' in s:
        return 'AMD Athlon'
    if 'm4' in s:
        return 'Apple M4'
    if 'm3' in s:
        return 'Apple M3'
    if 'm2' in s:
        return 'Apple M2'
    if 'm1' in s:
        return 'Apple M1'
    if 'snapdragon' in s:
        return 'Snapdragon'
    if 'mediatek' in s:
        return 'MediaTek'
    return 'Other'


def bucket_processor_tier(x):
    family = bucket_processor_family(x)
    if family in {'Intel Celeron', 'Intel Pentium', 'AMD Athlon', 'MediaTek'}:
        return 'Entry'
    if family in {'Intel Core i3', 'Intel Core 3', 'AMD Ryzen 3', 'Snapdragon'}:
        return 'Mainstream'
    if family in {'Intel Core i5', 'Intel Core 5', 'AMD Ryzen 5'}:
        return 'Upper mainstream'
    if family in {'Intel Core i7', 'Intel Core 7', 'AMD Ryzen 7', 'Intel Core Ultra 5'}:
        return 'Performance'
    if family in {'Intel Core i9', 'Intel Core 9', 'Intel Core Ultra 7', 'Intel Core Ultra 9', 'AMD Ryzen 9', 'AMD Ryzen AI 7', 'AMD Ryzen AI 9'}:
        return 'Premium performance'
    if family.startswith('Apple M'):
        return 'Apple silicon'
    return 'Other'


def bucket_ram(x):
    value = parse_ram_gb(x)
    if pd.isna(value):
        return 'Missing'
    if value <= 8:
        return '8 GB or less'
    if value <= 16:
        return '16 GB'
    return '24 GB or more'


def bucket_storage(x):
    value = parse_storage_gb(x)
    if pd.isna(value):
        return 'Missing'
    if value <= 256:
        return '256 GB or less'
    if value <= 512:
        return '512 GB'
    return '1 TB or more'


def bucket_warranty(x):
    value = parse_int(x)
    if pd.isna(value):
        return 'Missing'
    if value <= 1:
        return '1 year'
    if value == 2:
        return '2 years'
    return '3+ years'


def get_ordered_categories(values, preferred):
    present = [x for x in preferred if x in values]
    remainder = [x for x in values if x not in preferred]
    return present + remainder


def prepare_attribute_categories(df_in, attribute_cols):
    df_out = df_in.copy()
    preferred_orders = {
        'Company_Bucket_10': ['ASUS', 'Lenovo', 'HP', 'DELL', 'Acer'],
        'Price_Bucket': ['Budget', 'Mid-range', 'Premium'],
        'Processor_Tier': ['Entry', 'Mainstream', 'Upper mainstream', 'Performance', 'Premium performance', 'Apple silicon', 'Other'],
        'RAM_Bucket': ['8 GB or less', '16 GB', '24 GB or more'],
        'Storage_Bucket': ['256 GB or less', '512 GB', '1 TB or more'],
        'Warranty_Bucket': ['1 year', '2 years', '3+ years'],
    }
    for col in attribute_cols:
        values = df_out[col].dropna().astype(str).unique().tolist()
        ordered = get_ordered_categories(values, preferred_orders.get(col, []))
        df_out[col] = pd.Categorical(df_out[col], categories=ordered, ordered=True)
    return df_out


def build_design_matrix(df_in, attribute_cols):
    X = pd.get_dummies(df_in[attribute_cols], drop_first=True, prefix_sep='::').astype(float)
    X = sm.add_constant(X, has_constant='add').astype(float)
    return X


def fit_conjoint_like_model(df_in, attribute_cols, target_col, test_size=0.2, random_state=42):
    df_model = prepare_attribute_categories(df_in.copy(), attribute_cols)
    X = build_design_matrix(df_model, attribute_cols)
    y = pd.to_numeric(df_model[target_col], errors='coerce').astype(float)
    valid = y.notna() & X.notna().all(axis=1)
    X = X.loc[valid]
    y = y.loc[valid]
    df_model = df_model.loc[valid].copy()

    model = sm.OLS(y, X).fit(cov_type='HC3')
    idx_train, idx_test = train_test_split(X.index, test_size=test_size, random_state=random_state)
    model_train = sm.OLS(y.loc[idx_train], X.loc[idx_train]).fit()
    preds_test = model_train.predict(X.loc[idx_test])

    holdout_rmse = float(np.sqrt(np.mean((y.loc[idx_test] - preds_test) ** 2)))
    holdout_mae = float(np.mean(np.abs(y.loc[idx_test] - preds_test)))
    denom = np.sum((y.loc[idx_test] - y.loc[idx_test].mean()) ** 2)
    holdout_r2 = float(1 - np.sum((y.loc[idx_test] - preds_test) ** 2) / denom) if denom > 0 else np.nan

    coef_df = pd.DataFrame({'term': model.params.index, 'beta': model.params.values, 'p_value': model.pvalues.values})
    coef_df = coef_df[coef_df['term'] != 'const'].copy()
    coef_df[['attribute', 'level']] = coef_df['term'].str.split('::', n=1, expand=True)

    partworth_rows = []
    baseline_profile = {}
    for attr in attribute_cols:
        levels = list(df_model[attr].cat.categories)
        baseline = levels[0]
        baseline_profile[attr] = baseline
        raw_utils = {baseline: 0.0}
        for _, row in coef_df.loc[coef_df['attribute'] == attr].iterrows():
            raw_utils[row['level']] = row['beta']
        for level in levels:
            raw_utils.setdefault(level, 0.0)
        mean_util = np.mean(list(raw_utils.values()))
        for level in levels:
            partworth_rows.append({
                'attribute': attr,
                'level': level,
                'raw_partworth_vs_baseline': raw_utils[level],
                'zero_centered_partworth': raw_utils[level] - mean_util,
            })

    partworth_df = pd.DataFrame(partworth_rows)
    importance_df = partworth_df.groupby('attribute')['zero_centered_partworth'].agg(lambda s: s.max() - s.min()).reset_index(name='utility_range')
    importance_df['importance_pct'] = 100 * importance_df['utility_range'] / importance_df['utility_range'].sum()
    importance_df = importance_df.sort_values('importance_pct', ascending=False)

    fit_stats = pd.DataFrame([{
        'n_obs': int(model.nobs),
        'r_squared': float(model.rsquared),
        'adj_r_squared': float(model.rsquared_adj),
        'f_pvalue': float(model.f_pvalue) if model.f_pvalue is not None else np.nan,
        'holdout_r2': holdout_r2,
        'holdout_rmse': holdout_rmse,
        'holdout_mae': holdout_mae,
    }])

    return {
        'model': model,
        'X': X,
        'y': y,
        'coef_df': coef_df.sort_values(['attribute', 'beta'], ascending=[True, False]),
        'partworth_df': partworth_df.sort_values(['attribute', 'zero_centered_partworth'], ascending=[True, False]),
        'importance_df': importance_df,
        'fit_stats': fit_stats,
        'baseline_profile': baseline_profile,
    }

In [ ]:
def canonical_company_name(x):
    key = norm(x).lower()
    return COMPANY_NAME_MAP.get(key, norm(x))


raw_df = pd.read_csv(CSV_PATH)
feature_df = raw_df.copy()

required_cols = ['Company', 'Price', 'Processor', 'RAM', 'Storage', 'Warranty', 'Star_Rating']
missing_cols = [c for c in required_cols if c not in feature_df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

feature_df['Company'] = feature_df['Company'].apply(canonical_company_name)
feature_df = feature_df.loc[feature_df['Company'].isin(ALLOWED_COMPANIES)].copy()

feature_df['Price_num'] = feature_df['Price'].apply(parse_int)
feature_df['Star_Rating_num'] = feature_df['Star_Rating'].apply(parse_float)
feature_df['RAM_GB'] = feature_df['RAM'].apply(parse_ram_gb)
feature_df['Storage_GB'] = feature_df['Storage'].apply(parse_storage_gb)
feature_df['Warranty_Years'] = feature_df['Warranty'].apply(parse_int)

company_counts = feature_df['Company'].astype(str).str.strip().value_counts()
brands_10 = set(company_counts[company_counts >= MIN_BRAND_COUNT].index)
brands_major = sorted(company_counts[company_counts >= MAJOR_BRAND_COUNT].index.tolist())

feature_df['Company_Bucket_10'] = feature_df['Company']
feature_df['Price_Bucket'] = feature_df['Price'].apply(bucket_price)
feature_df['Processor_Tier'] = feature_df['Processor'].apply(bucket_processor_tier)
feature_df['RAM_Bucket'] = feature_df['RAM'].apply(bucket_ram)
feature_df['Storage_Bucket'] = feature_df['Storage'].apply(bucket_storage)
feature_df['Warranty_Bucket'] = feature_df['Warranty'].apply(bucket_warranty)

overall_df = feature_df.loc[
    feature_df['Star_Rating_num'].notna()
    & feature_df['Price_Bucket'].ne('Missing')
    & feature_df['Processor_Tier'].ne('Missing')
    & feature_df['RAM_Bucket'].ne('Missing')
    & feature_df['Storage_Bucket'].ne('Missing')
    & feature_df['Warranty_Bucket'].ne('Missing')
].copy()

print('Raw shape:', raw_df.shape)
print('Allowed-company shape:', feature_df.shape)
print('Overall modeling shape:', overall_df.shape)
print('Major brands for within-brand fits:', brands_major)
overall_df.head()

In [ ]:
overall_results = fit_conjoint_like_model(overall_df, CORE_ATTRIBUTES_OVERALL, TARGET_COL)
overall_results['fit_stats']

In [ ]:
overall_model = overall_results['model']
print(overall_model.summary())

overall_perf_panel = pd.DataFrame([{
    'n_obs': int(overall_model.nobs),
    'r_squared': float(overall_model.rsquared),
    'adj_r_squared': float(overall_model.rsquared_adj),
    'aic': float(overall_model.aic),
    'bic': float(overall_model.bic),
    'f_statistic': float(overall_model.fvalue) if overall_model.fvalue is not None else np.nan,
    'f_pvalue': float(overall_model.f_pvalue) if overall_model.f_pvalue is not None else np.nan,
    'holdout_r2': float(overall_results['fit_stats']['holdout_r2'].iat[0]),
    'holdout_rmse': float(overall_results['fit_stats']['holdout_rmse'].iat[0]),
    'holdout_mae': float(overall_results['fit_stats']['holdout_mae'].iat[0]),
}])
display(overall_perf_panel)
overall_perf_panel.to_csv(OUTPUT_DIR / 'overall_regression_performance.csv', index=False)

X_vif = overall_results['X'].drop(columns='const', errors='ignore').copy()
if X_vif.shape[1] > 0:
    overall_vif_df = pd.DataFrame({
        'term': X_vif.columns,
        'vif': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
    })
    overall_vif_df[['attribute', 'level']] = overall_vif_df['term'].str.split('::', n=1, expand=True)
    overall_vif_summary = overall_vif_df.groupby('attribute')['vif'].agg(['max', 'mean']).reset_index().rename(columns={'max': 'max_vif', 'mean': 'mean_vif'})
    display(overall_vif_df.sort_values('vif', ascending=False))
    display(overall_vif_summary.sort_values('max_vif', ascending=False))
    overall_vif_df.to_csv(OUTPUT_DIR / 'overall_vif_by_term.csv', index=False)
    overall_vif_summary.to_csv(OUTPUT_DIR / 'overall_vif_by_attribute.csv', index=False)
else:
    print('No encoded regressors available for VIF calculation.')


In [ ]:
overall_results['importance_df']

In [ ]:
overall_results['partworth_df']

In [ ]:
overall_results['coef_df'].to_csv(OUTPUT_DIR / 'overall_betas.csv', index=False)
overall_results['partworth_df'].to_csv(OUTPUT_DIR / 'overall_partworths.csv', index=False)
overall_results['importance_df'].to_csv(OUTPUT_DIR / 'overall_importance.csv', index=False)
overall_results['fit_stats'].to_csv(OUTPUT_DIR / 'overall_fit_stats.csv', index=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=overall_results['importance_df'], x='importance_pct', y='attribute', color='#264653')
plt.title('Overall Attribute Importance')
plt.xlabel('Importance %')
plt.ylabel('')
plt.tight_layout()
plt.show()

for attr in CORE_ATTRIBUTES_OVERALL:
    plot_df = overall_results['partworth_df'].loc[overall_results['partworth_df']['attribute'] == attr].copy()
    plt.figure(figsize=(9, 4))
    sns.barplot(data=plot_df, x='level', y='zero_centered_partworth', color='#2a9d8f')
    plt.xticks(rotation=30, ha='right')
    plt.title(f'Overall Zero-Centered Part-Worths: {attr}')
    plt.xlabel('')
    plt.ylabel('Zero-centered utility')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'overall_{attr.lower()}_partworths.png', dpi=180, bbox_inches='tight')
    plt.show()

In [ ]:
brand_results = {}
brand_fit_rows = []
brand_importance_rows = []
brand_partworth_rows = []

for brand in brands_major:
    brand_df = overall_df.loc[overall_df['Company'] == brand].copy()
    if len(brand_df) < 30:
        continue

    result = fit_conjoint_like_model(brand_df, CORE_ATTRIBUTES_WITHIN_BRAND, TARGET_COL)
    brand_results[brand] = result

    fit_row = result['fit_stats'].copy()
    fit_row.insert(0, 'brand', brand)
    brand_fit_rows.append(fit_row)

    brand_importance_rows.append(result['importance_df'].assign(brand=brand))
    brand_partworth_rows.append(result['partworth_df'].assign(brand=brand))

    result['coef_df'].to_csv(OUTPUT_DIR / f'{brand.lower()}_betas.csv', index=False)
    result['partworth_df'].to_csv(OUTPUT_DIR / f'{brand.lower()}_partworths.csv', index=False)
    result['importance_df'].to_csv(OUTPUT_DIR / f'{brand.lower()}_importance.csv', index=False)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=result['importance_df'], x='importance_pct', y='attribute', color='#457b9d')
    plt.title(f'{brand}: Attribute Importance')
    plt.xlabel('Importance %')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{brand.lower()}_importance.png', dpi=180, bbox_inches='tight')
    plt.show()

    for attr in CORE_ATTRIBUTES_WITHIN_BRAND:
        plot_df = result['partworth_df'].loc[result['partworth_df']['attribute'] == attr].copy()
        plt.figure(figsize=(9, 4))
        sns.barplot(data=plot_df, x='level', y='zero_centered_partworth', color='#e76f51')
        plt.xticks(rotation=30, ha='right')
        plt.title(f'{brand}: Zero-Centered Part-Worths - {attr}')
        plt.xlabel('')
        plt.ylabel('Zero-centered utility')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'{brand.lower()}_{attr.lower()}_partworths.png', dpi=180, bbox_inches='tight')
        plt.show()

brand_fit_summary = pd.concat(brand_fit_rows, ignore_index=True) if brand_fit_rows else pd.DataFrame()
brand_importance_compare = pd.concat(brand_importance_rows, ignore_index=True) if brand_importance_rows else pd.DataFrame()
brand_partworths_compare = pd.concat(brand_partworth_rows, ignore_index=True) if brand_partworth_rows else pd.DataFrame()

In [ ]:
brand_fit_summary

In [ ]:
if len(brand_fit_summary):
    brand_fit_summary.to_csv(OUTPUT_DIR / 'brand_fit_summary.csv', index=False)
    brand_importance_compare.to_csv(OUTPUT_DIR / 'brand_importance_compare.csv', index=False)
    brand_partworths_compare.to_csv(OUTPUT_DIR / 'brand_partworths_compare.csv', index=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=brand_importance_compare, x='importance_pct', y='attribute', hue='brand')
    plt.title('Attribute Importance by Brand')
    plt.xlabel('Importance %')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'brand_importance_compare.png', dpi=180, bbox_inches='tight')
    plt.show()

brand_importance_compare.head(20)

## Presentation-Ready Outputs

These cells mirror the summary and diagnostic outputs from the original workflow, using the fitted values from the current CSV.

In [ ]:
overall_model = overall_results['model']
overall_partworths = overall_results['partworth_df'].copy()
overall_importance = overall_results['importance_df'].copy()
overall_fit_stats = overall_results['fit_stats'].copy()

overall_level_support = []
for attr in CORE_ATTRIBUTES_OVERALL:
    tmp = overall_df[attr].value_counts(dropna=False).rename_axis('level').reset_index(name='n_products')
    tmp.insert(0, 'attribute', attr)
    overall_level_support.append(tmp)
overall_level_support = pd.concat(overall_level_support, ignore_index=True)

overall_best_levels = (
    overall_partworths.sort_values(['attribute', 'zero_centered_partworth'], ascending=[True, False])
    .groupby('attribute')
    .first()
    .reset_index()[['attribute', 'level', 'raw_partworth_vs_baseline', 'zero_centered_partworth']]
    .rename(columns={'level': 'best_level'})
)

overall_worst_levels = (
    overall_partworths.sort_values(['attribute', 'zero_centered_partworth'], ascending=[True, True])
    .groupby('attribute')
    .first()
    .reset_index()[['attribute', 'level', 'raw_partworth_vs_baseline', 'zero_centered_partworth']]
    .rename(columns={'level': 'worst_level'})
)

overall_best_worst = overall_best_levels.merge(
    overall_worst_levels[['attribute', 'worst_level', 'raw_partworth_vs_baseline', 'zero_centered_partworth']],
    on='attribute',
    suffixes=('_best', '_worst'),
)

overall_coef_plot_df = overall_results['coef_df'].copy()
if len(overall_coef_plot_df):
    conf = overall_model.conf_int()
    conf.columns = ['ci_low', 'ci_high']
    conf = conf.reset_index().rename(columns={'index': 'term'})
    overall_coef_plot_df = overall_coef_plot_df.merge(conf, on='term', how='left')

display(overall_fit_stats)
display(overall_importance)
display(overall_best_worst)
display(overall_level_support)

overall_fit_stats.to_csv(OUTPUT_DIR / 'ppt_overall_fit_summary.csv', index=False)
overall_importance.to_csv(OUTPUT_DIR / 'ppt_overall_attribute_importance.csv', index=False)
overall_best_worst.to_csv(OUTPUT_DIR / 'ppt_overall_best_worst_levels.csv', index=False)
overall_level_support.to_csv(OUTPUT_DIR / 'ppt_overall_level_support.csv', index=False)
overall_partworths.to_csv(OUTPUT_DIR / 'ppt_overall_partworths_full.csv', index=False)
if len(overall_coef_plot_df):
    overall_coef_plot_df.to_csv(OUTPUT_DIR / 'ppt_overall_coefficients_with_ci.csv', index=False)

preds_overall = overall_model.predict(overall_results['X'])
residuals_overall = overall_results['y'] - preds_overall
diag_df = pd.DataFrame({'actual_rating': overall_results['y'], 'predicted_rating': preds_overall, 'residual': residuals_overall})
diag_df.to_csv(OUTPUT_DIR / 'ppt_overall_prediction_diagnostics.csv', index=False)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=diag_df, x='actual_rating', y='predicted_rating', s=55, alpha=0.8)
line_min = min(diag_df['actual_rating'].min(), diag_df['predicted_rating'].min())
line_max = max(diag_df['actual_rating'].max(), diag_df['predicted_rating'].max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle='--', color='black', linewidth=1)
plt.title('Observed vs Predicted Rating')
plt.xlabel('Observed rating')
plt.ylabel('Predicted rating')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ppt_overall_observed_vs_predicted.png', dpi=200, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=diag_df, x='predicted_rating', y='residual', s=55, alpha=0.8)
plt.axhline(0, linestyle='--', color='black', linewidth=1)
plt.title('Residuals vs Predicted Rating')
plt.xlabel('Predicted rating')
plt.ylabel('Residual')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ppt_overall_residuals_vs_predicted.png', dpi=200, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(diag_df['residual'], bins=20, kde=True, color='#e76f51')
plt.title('Residual Distribution')
plt.xlabel('Residual')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ppt_overall_residual_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

if len(overall_coef_plot_df):
    coef_plot = overall_coef_plot_df.sort_values('beta', ascending=True).copy()
    plt.figure(figsize=(10, max(5, 0.35 * len(coef_plot))))
    plt.errorbar(
        x=coef_plot['beta'],
        y=np.arange(len(coef_plot)),
        xerr=[coef_plot['beta'] - coef_plot['ci_low'], coef_plot['ci_high'] - coef_plot['beta']],
        fmt='o',
        color='#264653',
        ecolor='#264653',
        capsize=3,
    )
    plt.yticks(np.arange(len(coef_plot)), coef_plot['term'])
    plt.axvline(0, linestyle='--', color='black', linewidth=1)
    plt.title('Overall Coefficients with 95% CI')
    plt.xlabel('Coefficient')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ppt_overall_coefficients_with_ci.png', dpi=200, bbox_inches='tight')
    plt.show()

if len(brand_results):
    brand_best_rows = []
    for brand, result in brand_results.items():
        tmp_best = (
            result['partworth_df']
            .sort_values(['attribute', 'zero_centered_partworth'], ascending=[True, False])
            .groupby('attribute')
            .first()
            .reset_index()[['attribute', 'level', 'zero_centered_partworth']]
        )
        tmp_best.insert(0, 'brand', brand)
        brand_best_rows.append(tmp_best)
    brand_best_levels_compare = pd.concat(brand_best_rows, ignore_index=True)
    brand_best_levels_compare.to_csv(OUTPUT_DIR / 'ppt_brand_best_levels_compare.csv', index=False)

ppt_summary_overall = overall_importance.copy()
ppt_summary_overall['importance_pct'] = ppt_summary_overall['importance_pct'].round(2)
ppt_top_levels = overall_partworths.sort_values('zero_centered_partworth', ascending=False).head(10)[['attribute', 'level', 'zero_centered_partworth']].reset_index(drop=True)
ppt_bottom_levels = overall_partworths.sort_values('zero_centered_partworth', ascending=True).head(10)[['attribute', 'level', 'zero_centered_partworth']].reset_index(drop=True)
ppt_top_levels.to_csv(OUTPUT_DIR / 'ppt_top_10_utility_levels.csv', index=False)
ppt_bottom_levels.to_csv(OUTPUT_DIR / 'ppt_bottom_10_utility_levels.csv', index=False)

display(ppt_summary_overall)
display(ppt_top_levels)
display(ppt_bottom_levels)

## Final Gradio Exports

This cell forces the final outputs into the exact `tab_2.csv` and `tab_3.csv` schemas used downstream.

In [ ]:
ATTRIBUTE_LABEL_MAP = {
    'Company_Bucket_10': 'Company',
    'Price_Bucket': 'Price',
    'Processor_Tier': 'Processor',
    'RAM_Bucket': 'RAM',
    'Storage_Bucket': 'Storage',
    'Warranty_Bucket': 'Warranty',
}


tab2_overall = overall_importance.copy()
tab2_overall['scope'] = 'Overall'
tab2_overall['attribute'] = tab2_overall['attribute'].map(ATTRIBUTE_LABEL_MAP)
tab2_overall = tab2_overall[['scope', 'attribute', 'utility_range', 'importance_pct']]

tab2_brand = brand_importance_compare.copy()
if len(tab2_brand):
    tab2_brand = tab2_brand.rename(columns={'brand': 'scope'})
    tab2_brand['attribute'] = tab2_brand['attribute'].map(ATTRIBUTE_LABEL_MAP)
    tab2_brand = tab2_brand[['scope', 'attribute', 'utility_range', 'importance_pct']]

tab_2_export = pd.concat([tab2_overall, tab2_brand], ignore_index=True) if len(tab2_brand) else tab2_overall.copy()
tab_2_export['scope'] = tab_2_export['scope'].replace({'Acer': 'Acer', 'ASUS': 'ASUS', 'DELL': 'DELL', 'HP': 'HP', 'Lenovo': 'Lenovo'})

tab3_overall = overall_partworths.copy()
tab3_overall['scope'] = 'Overall'
tab3_overall['attribute'] = tab3_overall['attribute'].map(ATTRIBUTE_LABEL_MAP)
tab3_overall = tab3_overall[['scope', 'attribute', 'level', 'zero_centered_partworth']]

tab3_brand = brand_partworths_compare.copy()
if len(tab3_brand):
    tab3_brand = tab3_brand.rename(columns={'brand': 'scope'})
    tab3_brand['attribute'] = tab3_brand['attribute'].map(ATTRIBUTE_LABEL_MAP)
    tab3_brand = tab3_brand[['scope', 'attribute', 'level', 'zero_centered_partworth']]

tab_3_export = pd.concat([tab3_overall, tab3_brand], ignore_index=True) if len(tab3_brand) else tab3_overall.copy()

tab_2_export['utility_range'] = tab_2_export['utility_range'].round(6)
tab_2_export['importance_pct'] = tab_2_export['importance_pct'].round(4)
tab_3_export['zero_centered_partworth'] = tab_3_export['zero_centered_partworth'].round(6)

tab_2_export.to_csv(OUTPUT_DIR / 'tab_2.csv', index=False)
tab_3_export.to_csv(OUTPUT_DIR / 'tab_3.csv', index=False)

print('Saved final files:')
print(OUTPUT_DIR / 'tab_2.csv')
print(OUTPUT_DIR / 'tab_3.csv')
display(tab_2_export.head())
display(tab_3_export.head())